In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import rootutils
from dotenv import load_dotenv

root = rootutils.setup_root(search_from='..', indicator="pyproject.toml", pythonpath=True, cwd=True)
print(f"Root directory: {root}")
print(f".env file path: {root / '.env'}")
load_dotenv(dotenv_path= root / ".env")

In [ ]:
from pydantic import SecretStr

from langchain_openai import ChatOpenAI

from src.sage.runtime import SageRuntime
from src.sage.types import SageRuntimeConfig
from src.tools.catalog import AVAILABLE_TOOLS
from src.agent.controller import AgentController, ControllerConfig
from src.utils.console_logging import ConsoleLogger

In [ ]:
logger = ConsoleLogger()

model = ChatOpenAI(
    model=f"gpt://{os.environ.get('YANDEX_FOLDER_ID')}/deepseek-v32/latest", #/gpt-oss-20b/latest",
    base_url = "https://ai.api.cloud.yandex.net/v1",
    api_key = SecretStr(os.environ.get('YANDEX_API_KEY', ".env hasn't been loaded or YANDEX_API_KEY is missing")),
    temperature = 0,
    timeout = 120,
    max_retries = 2,
)

sage_runtime_config = SageRuntimeConfig()
sage_runtime = SageRuntime(config=sage_runtime_config, logger=logger)

with open('prompts/sage_skill/v2.md', 'r') as f:
    sage_skill = f.read()

with open('prompts/system_prompt/v3.md', 'r') as f:
    system_prompt = f.read()

tools = [tool(sage_runtime, sage_skill if key == "sage_exec" else "") for key, tool in AVAILABLE_TOOLS.items()]

controller_config = ControllerConfig(
    max_steps=4,
    progress_logs=True,
)

controller = AgentController(model=model, config=controller_config, tools=tools, system_prompt=system_prompt, logger=logger)

In [ ]:
benchmark_path = root / "data/processed/agent_benchmark_5.json"

with open(benchmark_path, 'r') as f:
    benchmark_problems = json.load(f)['problems']

for i, problem in enumerate(benchmark_problems):
    logger.log(f"{problem['question']}", level=f"Problem {i}", color="bright_red", style="bold")
    logger.log(f"{problem['answer']}", level=f"Answer {i}", color="bright_green", style="bold")

In [ ]:
for i, problem in enumerate(benchmark_problems):
    if i == 0:
        continue
    logger.log(f"{problem['question']}", level=f"Problem {i}", color="bright_red", style="bold")
    result = controller.solve(problem['question'])
    logger.log(result.final_answer, level="Final Answer", color="bright_green", style="bold")